In [89]:
from pyspark.sql.functions import current_timestamp, to_timestamp, col

raw_path = "Files/raw/*.csv"

df_raw = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .csv(raw_path)
)

df_raw = df_raw.toDF(
    "SalesOrderNumber",
    "SalesOrderLineNumber",
    "OrderDate",
    "CustomerName",
    "EmailAddress",
    "ProductName",
    "OrderQuantity",
    "UnitPrice",
    "SalesAmount"
)

# convert order date
df_raw = df_raw.withColumn(
    "OrderDate",
    to_timestamp(col("OrderDate"))
)

# ✅ ADD ingestion column
df_raw = df_raw.withColumn(
    "IngestionDate",
    current_timestamp()
)

# overwrite bronze completely
df_raw.write.mode("overwrite").saveAsTable("bronze_sales")

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 91, Finished, Available, Finished, False)

In [90]:
df_clean = df_raw.withColumn(
    "IngestionDate",
    current_timestamp()
)
print(df_clean)

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 92, Finished, Available, Finished, False)

DataFrame[SalesOrderNumber: string, SalesOrderLineNumber: int, OrderDate: timestamp, CustomerName: string, EmailAddress: string, ProductName: string, OrderQuantity: int, UnitPrice: double, SalesAmount: double, IngestionDate: timestamp]


In [91]:
control_df = spark.sql("""
SELECT last_processed_date
FROM pipeline_control
WHERE pipeline_name = 'sales_pipeline'
""")
print(control_df)
last_date = control_df.collect()[0][0]

df_clean = spark.table("bronze_sales") \
    .filter(col("IngestionDate") > last_date)
    

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 93, Finished, Available, Finished, False)

DataFrame[last_processed_date: timestamp]


In [92]:
df_clean.write.mode("overwrite").saveAsTable("silver_sales")

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 94, Finished, Available, Finished, False)

In [93]:
df_clean.createOrReplaceTempView("updates")

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 95, Finished, Available, Finished, False)

In [94]:
%%sql
MERGE INTO silver_sales AS target
USING updates AS source
ON target.SalesOrderNumber = source.SalesOrderNumber
AND target.SalesOrderLineNumber = source.SalesOrderLineNumber

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *


StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 96, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [95]:
%%sql
UPDATE pipeline_control
SET last_processed_date = current_timestamp()
WHERE pipeline_name = 'sales_pipeline'


StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 97, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [96]:
df_clean.count()

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 98, Finished, Available, Finished, False)

32718

In [97]:
from pyspark.sql.functions import *

gold_df = (
    spark.table("silver_sales")
    .groupBy(
        to_date("OrderDate").alias("OrderDate"),
        col("ProductName")
    )
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("OrderQuantity").alias("TotalQuantity"),
        sum("SalesAmount").alias("TotalRevenue")
    )
)

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 99, Finished, Available, Finished, False)

In [98]:
gold_df.write.mode("overwrite").saveAsTable("gold_sales_summary")

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 100, Finished, Available, Finished, False)

In [99]:
gold_df.createOrReplaceTempView("gold_updates")

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 101, Finished, Available, Finished, False)

In [100]:
%%sql
MERGE INTO gold_sales_summary AS target
USING gold_updates AS source
ON target.OrderDate = source.OrderDate
AND target.ProductName = source.ProductName

WHEN MATCHED THEN UPDATE SET
    target.TotalOrders = source.TotalOrders,
    target.TotalQuantity = source.TotalQuantity,
    target.TotalRevenue = source.TotalRevenue

WHEN NOT MATCHED THEN INSERT *


StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 102, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [101]:
%%sql
SELECT *
FROM gold_sales_summary
ORDER BY OrderDate DESC
LIMIT 20;


StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 103, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 5 fields>

In [108]:
files = mssparkutils.fs.ls("Files/raw")

file_list = [f.path for f in files]

new_files = [f for f in file_list if f not in processed]

print("New files:", new_files)

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 110, Finished, Available, Finished, False)

New files: ['abfss://0b718c1b-6512-484a-8505-2fa1a78f4329@onelake.dfs.fabric.microsoft.com/9a9f1e59-0d35-419b-aded-91891a5284cb/Files/raw/2019.csv', 'abfss://0b718c1b-6512-484a-8505-2fa1a78f4329@onelake.dfs.fabric.microsoft.com/9a9f1e59-0d35-419b-aded-91891a5284cb/Files/raw/2020.csv', 'abfss://0b718c1b-6512-484a-8505-2fa1a78f4329@onelake.dfs.fabric.microsoft.com/9a9f1e59-0d35-419b-aded-91891a5284cb/Files/raw/2021.csv']


In [109]:
df_raw = (
    spark.read
    .option("header","false")
    .option("delimiter","\t")
    .schema(sales_schema)
    .csv(new_files)
)

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 111, Finished, Available, Finished, False)

In [110]:
from pyspark.sql import Row
from datetime import datetime

rows = [Row(file_name=f, processed_timestamp=datetime.now())
        for f in new_files]

spark.createDataFrame(rows) \
    .write.mode("append") \
    .saveAsTable("processed_files")

StatementMeta(, 8ab4738e-8118-4843-b613-b5dab1516345, 112, Finished, Available, Finished, False)

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS gold_sales (
    SalesOrderNumber STRING,
    SalesOrderLineNumber INT,
    OrderDate DATE,
    CustomerName STRING,
    EmailAddress STRING,
    ProductName STRING,
    OrderQuantity INT,
    UnitPrice DOUBLE,
    SalesAmount DOUBLE
);

In [ ]:
%%sql
MERGE INTO gold_sales AS target
USING silver_sales AS source
ON target.SalesOrderNumber = source.SalesOrderNumber
AND target.SalesOrderLineNumber = source.SalesOrderLineNumber

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *

In [ ]:
bad_records = incremental_df.filter(col("SalesAmount").isNull())

print("Bad records:", bad_records.count())